# Annexe B — Cahier de code, Chapitre 16
## Statistiques robustes

Ce notebook accompagne le chapitre 16 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Données avec une aberration franche
import numpy as np
x = np.array([10.0, 11.0, 9.5, 10.5, 100.0])    # le 100 est aberrant

## 16.1 — Médiane et point de rupture

In [ ]:
# la médiane ignore l'aberration, pas la moyenne
print("moyenne :", x.mean(), " médiane :", np.median(x))

## 16.2 — MAD

In [ ]:
# écart absolu médian → échelle robuste ; |z| = |x−méd| / (1.48·MAD)
from scipy.stats import median_abs_deviation
mad = median_abs_deviation(x, scale="normal")
z = np.abs(x - np.median(x)) / mad
print(z)                                  # le dernier point ressort

## 16.3 — M-estimateurs (Huber, Tukey)

In [ ]:
# fonctions de poids qui bornent l'influence des grosses erreurs (en NumPy)
from scipy.stats import median_abs_deviation
r = (x - np.median(x)) / median_abs_deviation(x, scale="normal")
c = 1.345
w_huber = np.where(np.abs(r) <= c, 1.0, c / np.abs(r))
c2 = 4.685
w_tukey = np.where(np.abs(r) <= c2, (1 - (r / c2) ** 2) ** 2, 0.0)
print("poids Huber :", w_huber)
print("poids Tukey :", w_tukey)        # 0 = point complètement rejeté

## 16.4 — IRLS

In [ ]:
# régression robuste y = a·t + b par moindres carrés repondérés itératifs
t = np.arange(20); y = 2.0 * t + 1 + np.random.randn(20)
y[10] = 80                             # une aberration au milieu
A = np.column_stack([t, np.ones_like(t)])
w = np.ones(len(y))
for _ in range(10):
    W = np.diag(w)
    beta = np.linalg.solve(A.T @ W @ A, A.T @ W @ y)   # ajustement pondéré
    resid = y - A @ beta
    s = 1.48 * np.median(np.abs(resid - np.median(resid))) + 1e-9
    w = np.minimum(1.0, 1.345 / (np.abs(resid / s) + 1e-9))  # poids Huber
print(beta)                            # ≈ [2, 1] : l'aberration est ignorée

## 16.5 — RANSAC

In [ ]:
# ajuste un modèle sur le plus grand consensus d'inliers
from skimage.measure import ransac, LineModelND
pts = np.column_stack([np.arange(20), np.arange(20) + np.random.randn(20)])
pts[5] = [5, 100]                         # aberration
modele, inliers = ransac(pts, LineModelND, min_samples=2,
                         residual_threshold=2, max_trials=100)
print(inliers.sum(), "inliers")

## 16.6 — Au-delà du RANSAC vanilla

In [ ]:
# variantes : estimateur de Theil-Sen (médiane des pentes), très robuste
from sklearn.linear_model import TheilSenRegressor
X = np.arange(20).reshape(-1, 1)
y = X.ravel() + np.random.randn(20); y[5] = 100
ts = TheilSenRegressor().fit(X, y)
print(ts.coef_, ts.intercept_)